Takes about 50s to load TTS model. This used to fire up right when we're importing the TTS module but right now it starts faster in about 10s.

In [2]:
from itertools import islice

import litellm

from main import TranslationOutput, message_content_from
from prompts import CAPTION_PROMPT
from util import read_jsonl


/home/maria/BookAudio/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/bin/sh: 1: sox: not found
SoX could not be found!

    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 


In [3]:
list(range(2,5))

[2, 3, 4]

- items_length = 20
- batch_size = 5
- i = 0
    - [0,1,2,3,4]
- i = 1
    - [0,1,2,3,4]
- i = 2
    - [0,1,2,3,4]
- i = 3
    - [1,2,3,4,5]

5/2=2.5

In [44]:
import math

def batch_indices(items_length, i, batch_size):
    assert batch_size <= items_length
    assert i >= 0
    assert i < items_length
    if i < batch_size / 2:
        return range(0, batch_size)
    elif i >= items_length - batch_size / 2:
        return range(items_length - batch_size, items_length)
    else:
        start = math.ceil(i - batch_size / 2)
        return range(start, start + batch_size)

for i in range(0, 15):
    print(i, list(batch_indices(15, i, 5)))


0 [0, 1, 2, 3, 4]
1 [0, 1, 2, 3, 4]
2 [0, 1, 2, 3, 4]
3 [1, 2, 3, 4, 5]
4 [2, 3, 4, 5, 6]
5 [3, 4, 5, 6, 7]
6 [4, 5, 6, 7, 8]
7 [5, 6, 7, 8, 9]
8 [6, 7, 8, 9, 10]
9 [7, 8, 9, 10, 11]
10 [8, 9, 10, 11, 12]
11 [9, 10, 11, 12, 13]
12 [10, 11, 12, 13, 14]
13 [10, 11, 12, 13, 14]
14 [10, 11, 12, 13, 14]


- 2791 with reasoning on takes about 21 hours
- Without reasoning it's about 1.5 hours
    - And it turned out to be 1.5 hours indeed

**BUT!!** You will have to manually reformat the JSON file because it just spurts out malformed JSON objects which isn't good.

In [60]:
from datetime import datetime

sentences = [x for x in read_jsonl("deepwork.jsonl") if x["type"] == "sentence"]
with open("deepwork-processed.jsonl", "w") as file:
    for (i, sentence) in enumerate(sentences):
        print("")
        time = datetime.now().strftime("%H:%M:%S")
        print(f"{time} Working with sentence {i+1}/{len(sentences)}")
        context = [sentences[i] for i in batch_indices(len(sentences), 8, 5)]
        # response = print(dict(
        response = litellm.completion(
            model="openai/unsloth/Qwen3.6-35B-A3B-MTP-GGUF:UD-Q4_K_M",
            api_base="http://localhost:8080/v1",
            api_key="not-needed",
            temperature=0.0,
            reasoning_effort="none",
            allowed_openai_params=['reasoning_effort'],
            messages=[{
                "role": "system",
                "content": CAPTION_PROMPT.format(
                    source_language="English",
                    target_language="Russian",
                ),
            }, {
                "role": "user",
                "content": f"""
        <context>
        {" ".join([sentence["content"] for sentence in context])}
        </context>

        <target>
        {sentence["content"]}
        </target>
                """
            }],
            response_format=TranslationOutput,
        )
        content = message_content_from(response) or "// No content //"
        file.write(content + "\n")




21:43:28 Working with sentence 1/2791

21:43:31 Working with sentence 2/2791

21:43:33 Working with sentence 3/2791

21:43:34 Working with sentence 4/2791

21:43:36 Working with sentence 5/2791

21:43:37 Working with sentence 6/2791

21:43:39 Working with sentence 7/2791

21:43:41 Working with sentence 8/2791

21:43:44 Working with sentence 9/2791

21:43:45 Working with sentence 10/2791

21:43:47 Working with sentence 11/2791

21:43:48 Working with sentence 12/2791

21:43:50 Working with sentence 13/2791

21:43:53 Working with sentence 14/2791

21:43:54 Working with sentence 15/2791

21:43:58 Working with sentence 16/2791

21:43:59 Working with sentence 17/2791

21:44:02 Working with sentence 18/2791

21:44:03 Working with sentence 19/2791

21:44:04 Working with sentence 20/2791

21:44:06 Working with sentence 21/2791

21:44:08 Working with sentence 22/2791

21:44:10 Working with sentence 23/2791

21:44:11 Working with sentence 24/2791

21:44:14 Working with sentence 25/2791

21:44:15

In [ ]:

# for (i, sentence) in enumerate(sentences):
#     generate_speech(sentence["content"], f"sentence-{i}.wav")

output = TranslationOutput.model_validate_json(
    content or f'// Error parsing: f{content} //'
)
output


Here we operate on a manually processed `deepwork-sentences.json`

In [4]:
import json

with open("deepwork-sentences.json") as f:
    data = json.load(f)


annotated_sentences = [
    TranslationOutput.model_validate(datum) for datum in data
] 
annotated_sentences

[TranslationOutput(sentences=[SentencePair(source='In the Swiss canton of St. Gallen,', target='В швейцарском кантоне Санкт-Галлен,', uncommon_words=[UncommonWord(source='canton', target='кантон')]), SentencePair(source='near the northern banks of Lake Zurich,', target='возле северных берегов Цюрихского озера,', uncommon_words=[UncommonWord(source='Zurich', target='Цюрих')]), SentencePair(source='is a village named Bollingen.', target='расположена деревня Бёллинген.', uncommon_words=[UncommonWord(source='Bollingen', target='Бёллинген')])]),
 TranslationOutput(sentences=[SentencePair(source='In 1922,', target='В 1922 году,', uncommon_words=[]), SentencePair(source='the psychiatrist Carl Jung', target='психиатр Карл Юнг', uncommon_words=[UncommonWord(source='psychiatrist', target='психиатр')]), SentencePair(source='chose this spot', target='выбрал это место', uncommon_words=[]), SentencePair(source='to begin building', target='чтобы начать строить', uncommon_words=[]), SentencePair(sourc

In [ ]:
from pydub import AudioSegment
from qwen import generate_audio
from datetime import datetime

instruct = (
    "Speak calmly in a neutral tone. At the end of every sentence "
    "lower your pitch noticeably downwards to signal a complete "
    "pause/closure. "
    # "Avoid any rising pitch or upward voice inflections "
    # "at sentence boundaries."
)
source_lang = "English"
target_lang = "Russian"

for i, annotated_sentence in enumerate(annotated_sentences):
    export_filename = f"output/wowsuper-{i}.mp3"
    time = datetime.now().strftime("%H:%M:%S")
    print(f"# {time} Sentence {i+1}/{len(annotated_sentences)}")

    total = AudioSegment.empty()
    for sentence in annotated_sentence.sentences:
        print(f"Generating audio: {(sentence.source, instruct, source_lang)}")
        total += generate_audio(sentence.source, instruct, source_lang)
        total += AudioSegment.silent(duration=400)
        print(f"Generating audio: {(sentence.target, instruct, target_lang)}")
        total += generate_audio(sentence.target, instruct, target_lang)
        for word in sentence.uncommon_words or []:
            total += AudioSegment.silent(duration=400)
            print(f"Generating audio: {(word.source, instruct, source_lang)}")
            total += generate_audio(word.source, instruct, source_lang)
            print(f"Generating audio: {(word.target, instruct, target_lang)}")
            total += generate_audio(word.target, instruct, target_lang)
        total += AudioSegment.silent(duration=800)

    total.export(export_filename, format="mp3")
    print(f"Written to {export_filename}")

# 16:44:55 Sentence 1/2791
Generating audio: ('In the Swiss canton of St. Gallen,', 'Speak calmly in a neutral tone. At the end of every sentence lower your pitch noticeably downwards to signal a complete pause/closure. ', 'English')


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 333.63it/s]
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Generating audio: ('В швейцарском кантоне Санкт-Галлен,', 'Speak calmly in a neutral tone. At the end of every sentence lower your pitch noticeably downwards to signal a complete pause/closure. ', 'Russian')


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Generating audio: ('canton', 'Speak calmly in a neutral tone. At the end of every sentence lower your pitch noticeably downwards to signal a complete pause/closure. ', 'English')


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


... (1,277 outputs left out)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Generating audio: ('loafing', 'Speak calmly in a neutral tone. At the end of every sentence lower your pitch noticeably downwards to signal a complete pause/closure. ', 'English')


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Generating audio: ('бездельничать, безделье', 'Speak calmly in a neutral tone. At the end of every sentence lower your pitch noticeably downwards to signal a complete pause/closure. ', 'Russian')


In [4]:
from qwen import generate_audio

instruct = (
    "Speak calmly in a neutral tone. At the end of every sentence "
    "lower your pitch noticeably downwards to signal a complete "
    "pause/closure. Avoid any rising pitch or upward voice inflections "
    "at sentence boundaries."
)

generate_audio("Больше все\u0301го на свете я лю\u0301блю око\u0301нное стекло", instruct, "Russian")

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


In [10]:
generate_audio("бо\u0301льше всего\u0301 на све\u0301те я люблю\u0301 око\u0301нное стекло\u0301", instruct, "Russian")

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


In [9]:
generate_audio("Большеоко\u0301нное стекло", instruct, "Russian")

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


In [17]:
ord("окоˊнное стекло"[3])

714